In [0]:
%sql
CREATE FOREIGN CATALOG IF NOT EXISTS pg_supplychain
  USING CONNECTION pg_supplychain_conn
  OPTIONS (database 'supplychain');

SELECT 'suppliers' t, COUNT(*) n FROM pg_supplychain.public.suppliers UNION ALL
SELECT 'products', COUNT(*) FROM pg_supplychain.public.products UNION ALL
SELECT 'purchase_orders', COUNT(*) FROM pg_supplychain.public.purchase_orders;

In [0]:
from pyspark.sql import functions as F
import time

SRC = "pg_supplychain.public"
TABLES = ["suppliers", "products", "purchase_orders"]
WM = "supplychainiq.ops.cdc_watermarks"
spark.sql(f"""CREATE TABLE IF NOT EXISTS {WM}
  (table_name STRING, last_updated_at TIMESTAMP, last_run_at TIMESTAMP, rows_pulled BIGINT)""")

In [0]:
%sql
SELECT 'suppliers' t, COUNT(*) AS changed_rows FROM pg_supplychain.public.suppliers       WHERE updated_at > '2014-05-23' UNION ALL
SELECT 'products',     COUNT(*)                 FROM pg_supplychain.public.products        WHERE updated_at > '2014-06-19' UNION ALL
SELECT 'purchase_orders', COUNT(*)              FROM pg_supplychain.public.purchase_orders WHERE updated_at > '2014-06-21';

In [0]:
def pull(t):
    row = spark.table(WM).where(F.col("table_name") == t).select("last_updated_at").first()
    wm = row[0] if row else None
    src = spark.table(f"{SRC}.{t}")
    if wm is not None:
        src = src.where(F.col("updated_at") > F.lit(wm))       # pushed down to Postgres as a WHERE clause

    batch_id = int(time.time() * 1000)
    (src.withColumn("_cdc_batch_id", F.lit(batch_id))
        .withColumn("_cdc_pulled_at", F.current_timestamp())
        .write.mode("append").option("mergeSchema", "true")
        .saveAsTable(f"supplychainiq.bronze.{t}_cdc"))

    s = (spark.table(f"supplychainiq.bronze.{t}_cdc").where(F.col("_cdc_batch_id") == batch_id)
           .agg(F.count("*").alias("n"), F.max("updated_at").alias("mx")).first())
    if s["n"] > 0:                                             # move the watermark only after a successful write
        spark.sql(f"""MERGE INTO {WM} w
            USING (SELECT :t AS table_name, CAST(:mx AS TIMESTAMP) AS last_updated_at,
                          current_timestamp() AS last_run_at, CAST(:n AS BIGINT) AS rows_pulled) s
            ON w.table_name = s.table_name
            WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""",
            args={"t": t, "mx": str(s["mx"]), "n": s["n"]})
    print(f"{t:>16}: watermark {wm} -> pulled {s['n']:,} rows (new max updated_at {s['mx']})")

for t in TABLES:
    pull(t)
display(spark.table(WM))

In [0]:
%sql
-- every observed version of a product, with the period it was valid
CREATE OR REPLACE TABLE supplychainiq.silver.products_scd2 AS
WITH v AS (SELECT DISTINCT item_id, dept_id, cat_id, supplier_id, unit_cost, status, updated_at
           FROM supplychainiq.bronze.products_cdc)
SELECT *, updated_at AS valid_from,
       LEAD(updated_at) OVER (PARTITION BY item_id ORDER BY updated_at)            AS valid_to,
       LEAD(updated_at) OVER (PARTITION BY item_id ORDER BY updated_at) IS NULL    AS is_current
FROM v;

-- latest state of each PO and supplier (a re-pulled row never creates a duplicate)
CREATE OR REPLACE TABLE supplychainiq.silver.purchase_orders AS
SELECT * EXCEPT (rn, _cdc_batch_id, _cdc_pulled_at) FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY po_id ORDER BY updated_at DESC, _cdc_pulled_at DESC) AS rn
  FROM supplychainiq.bronze.purchase_orders_cdc) WHERE rn = 1;

CREATE OR REPLACE TABLE supplychainiq.silver.suppliers AS
SELECT * EXCEPT (rn, _cdc_batch_id, _cdc_pulled_at) FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY supplier_id ORDER BY updated_at DESC, _cdc_pulled_at DESC) AS rn
  FROM supplychainiq.bronze.suppliers_cdc) WHERE rn = 1;

In [0]:
%sql
SELECT (SELECT COUNT(*) FROM pg_supplychain.public.purchase_orders)                         AS source_pos,
       (SELECT COUNT(*) FROM supplychainiq.silver.purchase_orders)                          AS silver_pos,
       (SELECT COUNT(*) FROM supplychainiq.silver.purchase_orders WHERE status = 'received') AS silver_received;

In [0]:
%sql
SELECT item_id, supplier_id, unit_cost, status, valid_from, valid_to, is_current
FROM supplychainiq.silver.products_scd2
WHERE item_id IN (SELECT item_id FROM supplychainiq.silver.products_scd2 GROUP BY item_id HAVING COUNT(*) > 1)
ORDER BY item_id, valid_from LIMIT 12;